# Lab 3.1 – Modular Skills with Pydantic

Refactor a mega-prompt DFD evaluator into single-responsibility skills with typed I/O and a registry that emits Vertex-compatible tool declarations.

In [ ]:
# --- Environment bootstrap (Colab + local + Docker) ---
import os, sys
from pathlib import Path

def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

REPO_URL = "https://github.com/fischer3-net/accurate_secure_rag_systems.git"
LAB_DIR = "labs/03-skills"

if _in_colab():
    REPO_ROOT = Path("/content/accurate_secure_rag_systems")
    if not REPO_ROOT.exists():
        get_ipython().system(f"git clone --depth 1 {REPO_URL} {REPO_ROOT}")
    LAB = REPO_ROOT / LAB_DIR
    os.chdir(LAB)
    sys.path.insert(0, str(LAB))
    sys.path.insert(0, str(REPO_ROOT / "labs" / "01-chunking"))
    get_ipython().run_line_magic("pip", "install -q pydantic python-dotenv langchain-text-splitters langchain-core pytest pyyaml pandas")
    print("Colab ready | LAB =", LAB)
else:
    LAB = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    if str(LAB) not in sys.path:
        sys.path.insert(0, str(LAB))
    print("Local ready | LAB =", LAB)


In [ ]:
import json, sys
from pathlib import Path

LAB = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(LAB))

from src.mega_prompt import mega_prompt_evaluate, mega_prompt_token_estimate
from src.registry import build_default_registry
from src.skills_syntax import validate_dfd_syntax
from src.skills_graph import check_trust_boundary_paths
from src.skills_policy import match_security_controls, set_policy_corpus, ControlCorpus
from src.skills_score import score_sdlc_compliance
from src.schemas import (
    ValidateDfdInput, CheckTrustPathsInput, MatchControlsInput, ScoreComplianceInput,
)

dfd = json.loads((LAB / "data" / "sample_dfd.json").read_text())
print("DFD:", dfd["name"])

## 1. Mega-prompt baseline (anti-pattern)

In [ ]:
blob = mega_prompt_evaluate(dfd)
print(f"Mega-prompt length: {len(blob)} chars ≈ {mega_prompt_token_estimate(dfd)} tokens")
print(blob[:400], "…")

## 2. Modular skills

In [ ]:
syn = validate_dfd_syntax(ValidateDfdInput.model_validate({"dfd": dfd}))
print("Syntax ok:", syn.ok, "| issues:", len(syn.issues))

paths = check_trust_boundary_paths(
    CheckTrustPathsInput.model_validate({"dfd": dfd, "require_crosses_trust_boundary": True})
)
print("Paths:", paths.summary)
for p in paths.paths:
    print(" ", " → ".join(p.path), "| controls:", p.governed_control_ids)

In [ ]:
corpus = ControlCorpus()
n = corpus.load_jsonl(LAB / "data" / "rag_chunks.jsonl")
set_policy_corpus(corpus)
print("Corpus rows:", n)

pol = match_security_controls(
    MatchControlsInput(query="external entity access to internal data stores", top_k=3)
)
for m in pol.matches:
    print(f"  {m.control_id}  score={m.score:.3f}  {m.section}")

In [ ]:
score = score_sdlc_compliance(
    ScoreComplianceInput(syntax=syn, trust_paths=paths, policy_matches=pol)
)
print("Overall:", score.overall_status)
print("Risk summary:", score.risk_summary)
for f in score.findings:
    print(" -", f)

## 3. Registry → Vertex tool declarations

In [ ]:
reg = build_default_registry()
decls = reg.vertex_tool_declarations()
print(f"{len(decls)} tools, ≈ {reg.estimate_tool_tokens()} tokens\n")
for d in decls:
    print(f"- {d['name']}: {d['description'][:80]}…")

## Notes for submission

- Which skills should stay deterministic in production?
- Where would you still use an LLM (if at all) inside a skill?
- How does structured output change testing compared with the mega-prompt?